# 📂 Document Loaders in LangChain

Simple, hands-on notebook to see how different Document Loaders work.

We'll use the sample files inside `sample_files/`:
- `sample.txt`
- `sample.pdf`
- `sample.csv`

Each loader converts its source into a **list of `Document` objects**, where every `Document` has:
- `page_content` → the actual text
- `metadata` → info about where it came from

Run the cells one by one 👇

## 🗃️ Types of Document Loaders Available in LangChain

LangChain doesn't have just one or two loaders — it has **hundreds** of them, grouped by the kind of source they read from. Here's the big picture before we touch any code:

| Category | Example Loaders | How it loads |
|---|---|---|
| **Plain files** | `TextLoader`, `CSVLoader`, `JSONLoader` | Opens the file directly from disk, reads its content, and converts it into `Document` object(s) |
| **PDFs** | `PyPDFLoader`, `PDFPlumberLoader`, `UnstructuredPDFLoader`, `AmazonTextractPDFLoader` | Parses the PDF's internal structure (text layer) page-by-page; scanned/complex PDFs need OCR-based loaders |
| **Folders of files** | `DirectoryLoader` | Scans a folder for files matching a pattern (e.g. `*.pdf`) and applies a chosen loader to each match |
| **Web pages** | `WebBaseLoader`, `SeleniumURLLoader`, `PlaywrightURLLoader` | Fetches the page over HTTP and parses the HTML (static pages); JS-heavy pages need a browser-based loader |
| **Cloud storage** | `S3FileLoader`, `AzureBlobStorageLoader`, `GoogleDriveLoader` | Authenticates with the cloud provider's API, downloads the file(s), then parses them like a local file |
| **Social platforms** | `TwitterTweetLoader`, `SlackDirectoryLoader`, `DiscordChatLoader` | Reads an exported data dump or calls the platform's API, then converts each post/message into a `Document` |
| **Messaging services** | `GMailLoader`, `WhatsAppChatLoader` | Parses an exported chat/email file or connects via API, one `Document` per message/email (or per conversation) |
| **Productivity tools** | `GitHubIssuesLoader`, `NotionDirectoryLoader`, `GoogleDocsLoader` | Calls the tool's API to pull pages/issues/docs and converts each into a `Document` with rich metadata |
| **Common file types** | `UnstructuredFileLoader`, `YoutubeLoader` | Detects the file/content type and routes it to the right internal parser automatically |

**Key idea:** no matter which category the loader belongs to, the *output* is always the same shape — a list of `Document` objects with `page_content` + `metadata`. That consistency is the whole point of Document Loaders.

> You don't need to memorize every loader. Get comfortable with the core ones (Text, PDF, CSV, Directory, Web) — for anything else, LangChain's docs will have a loader ready when you need it.

## 🔄 Where Document Loaders Fit in a RAG Pipeline

When you build a Retrieval-Augmented Generation (RAG) project, data doesn't start out as neat text — it starts as PDFs, CSVs, websites, Slack exports, etc. Here's what actually happens, step by step:

1. **Raw source** — you have a PDF, CSV, website, database, etc.
2. **Document Loader runs** — it reads that raw source and converts it into a standard list of `Document` objects (`page_content` + `metadata`), regardless of what the original format was.
3. **Text Splitter runs** — those `Document` objects (which can be huge) get broken into smaller chunks, because embedding models and LLM context windows can't handle arbitrarily large text.
4. **Embeddings are generated** — each chunk is converted into a numeric vector that captures its meaning.
5. **Vector Store saves it** — those vectors (with their original text + metadata) are stored in a vector database (e.g. FAISS, Chroma, Pinecone).
6. **Retriever + LLM** — at query time, the most relevant chunks are pulled back from the vector store and given to the LLM as context, so it can answer using your actual data instead of guessing.

So Document Loaders are literally **step 1** of this whole pipeline — if this step doesn't standardize the data properly, everything downstream (splitting, embedding, retrieval) becomes messy and inconsistent. That's why getting this step right matters so much.

## 0. Install dependencies

Run this once if these packages aren't already installed.

In [ ]:
%pip install -q langchain langchain-community pypdf

## 1. Text Loader

**Theory:**
- `TextLoader` reads a plain `.txt` file from disk.
- Since there's no natural way to split a `.txt` file (no pages, no rows), the **entire file becomes one single `Document` object**.
- Common use cases: log files, code snippets, transcripts, notes.

**Parameters used in the code below:**
- `file_path` (positional, required) → path to the `.txt` file we want to load. Here it's `"sample_files/sample.txt"`.
- `encoding` (optional, not used here but good to know) → lets you specify the file's character encoding (e.g. `"utf-8"`) if it's not the default.
- `autodetect_encoding` (optional, not used here) → if set to `True`, LangChain tries to auto-detect the encoding, useful when you're not sure what encoding a file uses.

After creating the loader, we call `.load()` — this is what actually reads the file and returns the list of `Document` objects.

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("sample_files/sample.txt")
text_docs = loader.load()

print(f"Total documents: {len(text_docs)}")
print("\n=== PAGE CONTENT ===")
print(text_docs[0].page_content)

print("\n=== METADATA ===")
print(text_docs[0].metadata)

## 2. PDF Loader (PyPDFLoader)

**Theory:**
- `PyPDFLoader` uses the `pypdf` library under the hood to read a PDF's text layer.
- Unlike `TextLoader`, a PDF has natural boundaries — its pages — so **each page becomes its own `Document` object**. A 3-page PDF gives you 3 `Document`s.
- Works well for text-based PDFs. For scanned images or complex layouts, you'd need something like `UnstructuredPDFLoader` or `AmazonTextractPDFLoader` (OCR-based).

**Parameters used in the code below:**
- `file_path` (positional, required) → path to the PDF file, here `"sample_files/sample.pdf"`.
- `extract_images` (optional, not used here, default `False`) → if `True`, tries to run OCR on embedded images inside the PDF to extract any text from them too.
- `password` (optional, not used here) → lets you unlock a password-protected PDF.

Each item in the returned list has metadata like `{'source': ..., 'page': 0}` — the `page` number tells you exactly where that chunk of text came from in the original PDF.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("sample_files/sample.pdf")
pdf_docs = loader.load()

print(f"Total pages: {len(pdf_docs)}")

for i, page in enumerate(pdf_docs):
    print(f"\n=== PAGE {i+1} ===")
    print(page.page_content[:150], "...")
    print("Metadata:", page.metadata)

## 3. CSV Loader

**Theory:**
- `CSVLoader` reads a `.csv` file row by row.
- **Each row becomes its own `Document` object** — the row's columns and values are stringified into `page_content` as `"column: value"` pairs, one per line.
- Great for tabular datasets, employee records, product catalogs, etc., where each row is a self-contained unit of information.

**Parameters used in the code below:**
- `file_path` (positional, required) → path to the CSV file, here `"sample_files/sample.csv"`.
- `csv_args` (optional, not used here) → a dict to control how the CSV is parsed, e.g. `{"delimiter": ";"}` if your file isn't comma-separated.
- `source_column` (optional, not used here) → lets you pick a specific column's value to use as the `source` in metadata, instead of the file path.
- `metadata_columns` (optional, not used here) → a list of column names you want pulled out into `metadata` instead of being part of the main `page_content`.

Metadata for each row automatically includes the `row` number, e.g. `{'source': ..., 'row': 0}`.

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("sample_files/sample.csv")
csv_docs = loader.load()

print(f"Total rows: {len(csv_docs)}")

print("\n=== ROW 1 CONTENT ===")
print(csv_docs[0].page_content)

print("\n=== ROW 1 METADATA ===")
print(csv_docs[0].metadata)

print("\n=== FIRST 3 ROWS (preview) ===")
for i, row in enumerate(csv_docs[:3]):
    print(f"Row {i+1}: {row.page_content[:80]}...")

## 4. Directory Loader

**Theory:**
- `DirectoryLoader` doesn't parse files itself — it **scans a folder**, finds files matching a pattern, and hands each matching file off to a loader class you specify.
- This is how you load many files (of the same type) in one go instead of writing a loop yourself.
- Loading a huge number of large files this way (eagerly) can use a lot of memory — that's where lazy loading (next section) helps.

**Parameters used in the code below:**
- `path` (required) → the root folder to scan, here `"sample_files"`.
- `glob` (required) → the file pattern to match, here `"**/*.pdf"` means "any `.pdf` file, in this folder or any subfolder."
- `loader_cls` (required) → which loader class to apply to every matched file — here we tell it to use `PyPDFLoader` for each PDF found.
- `show_progress` (optional) → if `True`, shows a progress bar while loading, handy when there are many files.
- `use_multithreading` (optional, not used here) → if `True`, loads multiple files in parallel to speed things up.
- `silent_errors` (optional, not used here) → if `True`, skips files that fail to load instead of stopping the whole run.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    path="sample_files",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

dir_docs = loader.load()

print(f"Total documents loaded: {len(dir_docs)}")
for doc in dir_docs:
    print(f"Source: {doc.metadata['source']}, Page: {doc.metadata.get('page', 'N/A')}")

## 5. Bonus: Eager vs Lazy Loading

**Theory:**
- **Eager loading (`.load()`)** → reads everything right away and returns a full list of `Document` objects in memory. Simple, but can be memory-heavy for large datasets.
- **Lazy loading (`.lazy_load()`)** → returns a generator that loads and yields **one `Document` at a time**, only when you ask for the next one. This keeps memory usage low and lets you start processing before everything is loaded.

**Parameters used in the code below:**
- Same `file_path` as before — the difference is purely in *which method* we call (`.load()` vs `.lazy_load()`), not in the constructor arguments.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Eager loading
loader = PyPDFLoader("sample_files/sample.pdf")
eager_pages = loader.load()
print(f"Eager loaded: {len(eager_pages)} pages")

# Lazy loading
loader = PyPDFLoader("sample_files/sample.pdf")
count = 0
for page in loader.lazy_load():
    count += 1
    print(f"Lazy loaded page {count}: {page.metadata}")

print(f"\nTotal lazy loaded: {count} pages")

## ✅ Summary

| Loader | Best for | Output |
|---|---|---|
| TextLoader | `.txt` files | 1 Document per file |
| PyPDFLoader | PDF files | 1 Document per page |
| CSVLoader | CSV files | 1 Document per row |
| DirectoryLoader | Entire folders | Multiple Documents |

Every loader gives back a **list of `Document` objects** — same structure regardless of source. That's what makes the rest of the RAG pipeline (splitting, embedding, retrieval) work smoothly no matter where the data came from.